# CSIRO Biomass image inference

Generate `submission.csv` from a locally trained model uploaded to a Kaggle Dataset.

In [ ]:
import os
import sys
import glob
import joblib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel, CLIPProcessor, CLIPModel

# ==========================================
# 設定
# ==========================================
COMP_DIR = "/kaggle/input/csiro-biomass"
DATASET_DIR = "/kaggle/input/koro2jp"

if DATASET_DIR not in sys.path:
    sys.path.append(DATASET_DIR)

TARGET_NAMES = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']
MAX_VALS = np.array([71.7865, 83.8407, 157.9836, 185.70, 157.9836])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ==========================================
# 1. モデルロード関数
# ==========================================
def load_model_from_input(target_name):
    print(f"Searching model for: {target_name} ...")
    
    # CLIP検索
    if "clip" in target_name.lower():
        patterns = [
            "/kaggle/input/*openai*clip*vit*large*336*",
            "/kaggle/input/*clip*vit*large*336*",
            "/kaggle/input/*clip*vit*large*",
            "/kaggle/input/*openai*clip*"
        ]
        candidates = []
        for pat in patterns:
            candidates.extend(glob.glob(pat))
        
        for path in candidates:
            if os.path.exists(os.path.join(path, "config.json")) or \
               glob.glob(os.path.join(path, "*", "config.json")):
                if not os.path.exists(os.path.join(path, "config.json")):
                    sub = glob.glob(os.path.join(path, "*", "config.json"))
                    if sub: path = os.path.dirname(sub[0])
                print(f"Found CLIP at: {path}")
                return path
        raise FileNotFoundError("CLIP model dataset not found in Input!")

    # その他
    candidates = glob.glob(f"/kaggle/input/*{target_name}*")
    model_path = None
    for path in candidates:
        if os.path.exists(os.path.join(path, "config.json")):
            model_path = path; break
        sub = glob.glob(os.path.join(path, "*", "config.json"))
        if sub: model_path = os.path.dirname(sub[0]); break
    
    if model_path is None: 
        raise FileNotFoundError(f"Model {target_name} not found.")
        
    print(f"Loading {target_name} from: {model_path}")
    return model_path

# ==========================================
# 2. 特徴量抽出関数 (No TTA, CLS+AVG)
# ==========================================
def get_embeddings_v4(image_paths, model_path, model_type="siglip", batch_size=32):
    print(f"Loading {model_type} model from {model_path}...")
    
    if model_type == "clip":
        processor = CLIPProcessor.from_pretrained(model_path)
        model = CLIPModel.from_pretrained(model_path).to(DEVICE).eval()
    else:
        processor = AutoImageProcessor.from_pretrained(model_path)
        model = AutoModel.from_pretrained(model_path).to(DEVICE).eval()

    embeddings = []
    
    # TTAなし
    for i in tqdm(range(0, len(image_paths), batch_size), desc=f"Extract {model_type}"):
        batch_paths = image_paths[i : i + batch_size]
        images = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                images.append(img)
            except:
                images.append(Image.new("RGB", (224, 224)))
        
        if not images: continue
        
        with torch.no_grad():
            if model_type == "clip":
                inputs = processor(images=images, return_tensors="pt", padding=True).to(DEVICE)
                out = model.get_image_features(**inputs)
            else:
                inputs = processor(images=images, return_tensors="pt").to(DEVICE)
                if model_type == "siglip":
                    out = model.get_image_features(**inputs)
                else: 
                    # DINOv2: CLS + AVG Pooling
                    outputs = model(**inputs)
                    last_hidden = outputs.last_hidden_state
                    cls_token = last_hidden[:, 0, :]
                    avg_pool = last_hidden[:, 1:, :].mean(dim=1)
                    out = torch.cat([cls_token, avg_pool], dim=1)

        emb = out / out.norm(dim=-1, keepdim=True)
        embeddings.append(emb.cpu().numpy())
        
    return np.vstack(embeddings)

# ==========================================
# 3. 後処理
# ==========================================
def post_process(df):
    cols = ["Dry_Green_g", "Dry_Clover_g", "Dry_Dead_g", "GDM_g", "Dry_Total_g"]
    if not all(c in df.columns for c in cols): return df
    
    # Cloverを少し下げる
    if "Dry_Clover_g" in df.columns:
        df["Dry_Clover_g"] *= 0.85
        
    Y = df[cols].values.T
    C = np.array([[1,1,0,-1,0], [0,0,1,1,-1]])
    P = np.eye(5) - C.T @ np.linalg.inv(C @ C.T) @ C
    Y_rec = (P @ Y).T.clip(min=0)
    df[cols] = Y_rec
    return df

# ==========================================
# Main
# ==========================================
test_df = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
unique_images = test_df[['image_path']].drop_duplicates().reset_index(drop=True)
full_paths = [os.path.join(COMP_DIR, p) for p in unique_images['image_path']]
print(f"Test Images: {len(full_paths)}")

# 1. 特徴量抽出
print("Extracting features (v4)...")
sig_path = load_model_from_input("siglip")
emb_sig = get_embeddings_v4(full_paths, sig_path, "siglip") 
dino_path = load_model_from_input("dinov2") 
emb_dino = get_embeddings_v4(full_paths, dino_path, "dinov2") 
clip_path = load_model_from_input("clip") 
emb_clip = get_embeddings_v4(full_paths, clip_path, "clip") 

print("Features extracted.")

# 2. 推論
models_to_run = ["ridge", "lgbm", "xgb", "cat"]

MODEL_WEIGHTS = {
    "ridge": 0.40, 
    "lgbm":  0.20, 
    "xgb":   0.20,
    "cat":   0.20
}

final_pred_accum = np.zeros((len(emb_sig), 5))
total_weight = 0

print("-" * 30)
print("Start Predicting (Target: detailed_ensemble)...")

for keyword in models_to_run:
    pat = os.path.join(DATASET_DIR, "**", "detailed_ensemble", keyword, "models_fold_*.pkl")
    files = glob.glob(pat, recursive=True)
    valid_dirs = list(set([os.path.dirname(f) for f in files]))
    
    if not valid_dirs:
        print(f"⚠️ {keyword} not found in detailed_ensemble, searching all...")
        pat = os.path.join(DATASET_DIR, "**", keyword, "models_fold_*.pkl")
        files = glob.glob(pat, recursive=True)
        valid_dirs = list(set([os.path.dirname(f) for f in files]))
        better_dirs = [d for d in valid_dirs if "detailed" in d or "v4" in d]
        if better_dirs: valid_dirs = better_dirs

    if not valid_dirs: 
        print(f"Skipping {keyword} (Not found)")
        continue
    
    keyword_pred_accum = np.zeros((len(emb_sig), 5))
    keyword_models_count = 0
    
    for model_dir in valid_dirs:
        print(f"  -> Loading {os.path.basename(model_dir)}...")
        
        dir_pred_accum = np.zeros((len(emb_sig), 5))
        folds = 0
        success_folds = 0
        
        for fold in range(5):
            ep = os.path.join(model_dir, f"engine_fold_{fold}.pkl")
            mp = os.path.join(model_dir, f"models_fold_{fold}.pkl")
            if not os.path.exists(ep) or not os.path.exists(mp): continue
            
            try:
                eng = joblib.load(ep)
                ms = joblib.load(mp) 
                
                # 特徴量結合 (3968次元)
                X_in = np.hstack([emb_sig, emb_dino, emb_clip])
                X_eng = eng.transform(X_in)
                
                p_fold = []
                if isinstance(ms, list): # GBDT List
                    for i, m in enumerate(ms):
                        if getattr(m, "_n_classes", None) is None: m._n_classes = 1
                        p = m.predict(X_eng)
                        if p.ndim > 1: p = p.flatten()
                        p = p * MAX_VALS[i]
                        p_fold.append(p)
                    dir_pred_accum += np.column_stack(p_fold)
                else: # RidgeCV
                    p = ms.predict(X_eng)
                    p = p * MAX_VALS
                    dir_pred_accum += p
                
                success_folds += 1
            except Exception as e:
                pass
        
        if success_folds > 0:
            keyword_pred_accum += (dir_pred_accum / success_folds)
            keyword_models_count += 1
            
    # そのモデル種別(xgbなど)の平均算出
    if keyword_models_count > 0:
        avg_pred = keyword_pred_accum / keyword_models_count
        weight = MODEL_WEIGHTS.get(keyword, 0)
        final_pred_accum += avg_pred * weight
        total_weight += weight
        print(f"✅ Added {keyword} to ensemble (Weight: {weight})")

if total_weight > 0:
    final_pred = final_pred_accum / total_weight
else:
    final_pred = np.zeros((len(emb_sig), 5))

pred_df = pd.DataFrame(final_pred, columns=TARGET_NAMES)
pred_df["image_path"] = unique_images["image_path"]
pred_df = post_process(pred_df)

sub = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
long_df = pred_df.melt(id_vars=["image_path"], value_vars=TARGET_NAMES, var_name="target_name", value_name="pred")
final = pd.merge(sub[["sample_id", "image_path", "target_name"]], long_df, on=["image_path", "target_name"], how="left")
final["target"] = final["pred"].fillna(0)
final[["sample_id", "target"]].to_csv("submission.csv", index=False)
print("Done! Submission saved.")